# 1. Import Lib

In [1]:
# Code to import libraries as you need in this assessment, e.g.,
import os
import re
import textwrap
import pandas as pd
from collections import Counter
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
import nltk
import re
from spellchecker import SpellChecker
from rapidfuzz import fuzz
from wordfreq import word_frequency
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from math import ceil
from tqdm.auto import tqdm



# 2. Helpers

In [2]:
TOKENIZER_PATTERN = re.compile(r"[a-zA-Z]+(?:[-'][a-zA-Z]+)?") # From documentation
STOPWORDS_PATH = "./data/stopwords_en.txt"
ENHANCED_PATH = "./data/stopwords_en_enhancement.txt" 
VOCAB_PATH = "./data/vocab.txt"
OUTPUT_PATH = "./data/processed.csv"
DATA_PATH = "./data/cosmetics_beauty_products_reviews.csv"

In [3]:
def load_stopwords(path: str) -> list[str]:
    with open(path, "r", encoding="utf-8") as f:
        words: list[str] = [line.strip().lower() for line in f if line.strip()]
    return words


def save_vocab(processed_docs: list[list[str]], path: str) -> dict:
    """
    Build a vocabulary from a list of tokenized documents.
    Returns: dict with word:index pairs, sorted alphabetically.
    """
    vocab_set = {t for doc in processed_docs for t in doc}
    vocab_sorted = sorted(vocab_set) # # A–Z (ASCII-ish; digits would sort before letters if any)
    lines = []
    for i, word in enumerate(vocab_sorted):
        lines.append(f"{word}:{i}")
    
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")
    print(f"\n[Saved] {len(lines)} words → {path}")

# 3. Main Logic
## 3.1. Examining and loading data
- Examine the data and explain your findings
- Load the data into proper data structures and get it ready for processing.

In [4]:
# Code to inspect the provided data file...
df_raw: pd.DataFrame = pd.read_csv(DATA_PATH)

In [5]:
df_raw.head(2)

,product_id,brand_name,review_id,review_title,review_text,author,review_date,review_rating,is_a_buyer,product_title,price,avg_product_rating,product_rating_count,product_tags,product_url
0,781070,Olay,16752142,Worth buying 50g one,Works as it claims. Could see the difference f...,Ashton Dsouza,23/01/2021 15:17,5.0,True,Olay Ultra Lightweight Moisturiser: Luminous W...,1599,4.1,43,NaN,https://www.nykaa.com/olay-ultra-lightweight-m...
1,781070,Olay,14682550,Best cream to start ur day,It does what it claims . Best thing is it smoo...,Amrit Neelam,07/09/2020 15:30,5.0,True,Olay Ultra Lightweight Moisturiser: Luminous W...,1599,4.1,43,NaN,https://www.nykaa.com/olay-ultra-lightweight-m...


In [6]:
df_raw.shape

(61284, 15)

## 3.2. Pre-processing data
Perform the required text pre-processing steps.

In [7]:
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

True

In [ ]:
def _split_tokens(text: str) -> list[str]:
    return TOKENIZER_PATTERN.findall(str(text))


def _process_token(t: str, stop_set: set[str]) -> str | None:
    if t:
        if len(t) >= 2:
            if t not in stop_set:
                return t
            else:
                return None
        else:
            return None
    else:
        return None
    

def _is_meaningful(word) -> bool:
    # 2-char words must be genuinely common (by/as/in pass, ab/ac fail)
    if len(word) == 2 and word_frequency(word, 'en') < 1e-4:
        return False
    return True
    

def check_and_fix_old(lemmatizer, speller: SpellChecker, word: str) -> str | None:
    if not word:
        return None

    word = lemmatizer.lemmatize(word, pos=wordnet.NOUN)

    c_light = re.sub(r'(.)\1{2,}', r'\1', word)
    c_hard  = re.sub(r'(.)\1+',    r'\1', word)

    # Step 1: known word → trust it, optionally normalize via correction
    if speller.known([word]):
        if not _is_meaningful(word):
            return None
        correction = speller.correction(word)
        if correction and _is_meaningful(correction) and fuzz.ratio(word, correction) >= 70:
            return correction
        return word  # word is valid — keep it even if no better correction found

    # Step 2: unknown → try collapsed variants to recover real base word
    for candidate in dict.fromkeys([c_hard, c_light, word]):
        if not candidate or not _is_meaningful(candidate):
            continue
        correction = speller.correction(candidate)
        if not correction or not _is_meaningful(correction):
            continue
        if fuzz.ratio(candidate, correction) >= 70:
            return correction

    return None


def check_and_fix(lemmatizer, speller: SpellChecker, word: str) -> str | None:
    if not word:
        return None

    c_light = re.sub(r'(.)\1{2,}', r'\1', word)
    c_hard  = re.sub(r'(.)\1+',    r'\1', word)

    if speller.known([word]):
        if not _is_meaningful(word):
            return None
        correction = speller.correction(word)
        if correction and _is_meaningful(correction) and fuzz.ratio(word, correction) >= 70:
            return lemmatizer.lemmatize(correction, pos=wordnet.NOUN)
        return lemmatizer.lemmatize(word, pos=wordnet.NOUN)

    for candidate in dict.fromkeys([c_hard, c_light, word]):
        if not candidate or not _is_meaningful(candidate):
            continue
        correction = speller.correction(candidate)
        if not correction or not _is_meaningful(correction):
            continue
        if fuzz.ratio(candidate, correction) >= 70:
            return lemmatizer.lemmatize(correction, pos=wordnet.NOUN)

    return None

def _filter_by_frequency(docs: list[list[str]], top_k_df: int = 20, min_tf: int = 2) -> list[list[str]]:
    tf: Counter[str] = Counter()
    df: Counter[str] = Counter()
    for doc in docs:
        tf.update(doc)
        df.update(set(doc))

    # min_tf=1 keeps everything; min_tf=2 (default) removes hapax legomena
    docs = [[t for t in doc if tf[t] >= min_tf] for doc in docs]

    top_df_words: set[str] = {w for w, _ in df.most_common(top_k_df)}
    docs = [
        filtered if (filtered := [t for t in doc if t not in top_df_words]) else doc
        for doc in docs
    ]
    return docs

In [10]:
def _preprocess_chunk(
    chunk: list[str],
    stop_set: set[str],
    worker_id: int,
) -> list[list[str]]:
    lemmatizer = WordNetLemmatizer()
    speller = SpellChecker()
    docs = []
    for text in tqdm(chunk, desc=f"Core {worker_id + 1}", position=worker_id, leave=True):
        tokens = []
        for token in _split_tokens(text):
            t = token.lower()
            t = _process_token(t, stop_set)
            t = check_and_fix(lemmatizer, speller, t)
            # post process again
            t = _process_token(t, stop_set)
            if t:
                tokens.append(t)
        docs.append(tokens)
    return docs
    

def preprocess(
    texts: list[str],
    stopwords: list[str],
    top_k_df: int = 20,
    min_tf: int = 2,
    n_jobs: int = -1,  # -1 = all available cores
) -> list[list[str]]:
    stop_set: set = set(w.lower().strip() for w in stopwords if w.strip())

    n_workers: int = os.cpu_count() if n_jobs == -1 else min(n_jobs, os.cpu_count())
    n_workers = max(1, n_workers)

    chunk_size: int = ceil(len(texts) / n_workers)
    chunks: list[list[str]] = [texts[i : i + chunk_size] for i in range(0, len(texts), chunk_size)]
    actual_workers: int = len(chunks)  # may be < n_workers for tiny inputs

    ordered: list[list[list[str]]] = [None] * actual_workers
    with ThreadPoolExecutor(max_workers=actual_workers) as executor:
        futures: dict = {
            executor.submit(_preprocess_chunk, chunk, stop_set, i): i
            for i, chunk in enumerate(chunks)
        }
        for future in as_completed(futures):
            idx = futures[future]
            ordered[idx] = future.result()

    docs: list[list[str]] = [doc for chunk_docs in ordered for doc in chunk_docs]

    tf: Counter[str] = Counter()
    df: Counter[str] = Counter()
    for doc in docs:
        tf.update(doc)
        df.update(set(doc))

    docs = [[t for t in doc if tf[t] >= min_tf] for doc in docs]

    top_df_words = {w for w, _ in df.most_common(top_k_df)}
    docs = [
        filtered if (filtered := [t for t in doc if t not in top_df_words]) else doc
        for doc in docs
    ]
    return docs

In [11]:
stopwords = load_stopwords(ENHANCED_PATH)  # or STOPWORDS_PATH
processed_docs = preprocess(
    df_raw["review_text"].fillna("").tolist(),
    stopwords,
    min_tf=1,   # no tf filtering on tiny corpus
    n_jobs=4,
)


Core 2:   0%|          | 0/15321 [00:00<?, ?it/s]

Core 3:   0%|          | 0/15321 [00:00<?, ?it/s]

Core 1:   0%|          | 0/15321 [00:00<?, ?it/s]

Core 4:   0%|          | 0/15321 [00:00<?, ?it/s]

In [13]:
processed_docs

[['work', 'claim', 'difference', 'day', 'play', 'cleanser', 'result'],
 ['claim', 'thing', 'smoothens', 'soft'],
 ['using',
  'month',
  'combination',
  'oily',
  'greasy',
  'absorbs',
  'quickly',
  'moisturizes',
  'work',
  'winter'],
 ['oily',
  'whip',
  'act',
  'great',
  'base',
  'primer',
  'smoothens',
  'moisturize',
  'felt',
  'needed',
  'moisturizer',
  'worth',
  'price',
  'buying'],
 ['please', 'refresh', 'try'],
 ['dry',
  'play',
  'representative',
  'suggest',
  'buy',
  'type',
  'please',
  'oily',
  'type',
  'don',
  'buy',
  'dry',
  'normal',
  'packaging',
  'box',
  'big',
  'size',
  'inside',
  'size',
  'cream',
  'great',
  'loss',
  'small',
  'quantity',
  'heavy',
  'price'],
 ['cream',
  'awesome',
  'rough',
  'soft',
  'smooth',
  'leaving',
  'oily',
  'effect',
  'continue'],
 ['instantly', 'tone', 'appearance'],
 ['eye',
  'cream',
  'combo',
  'effective',
  'work',
  'fine',
  'line',
  'eye',
  'dark',
  'circle',
  'worth',
  'penny'],


In [57]:
processed_docs

[['work',
  'claim',
  'difference',
  'day',
  'use',
  'play',
  'cleanser',
  'best',
  'result'],
 ['claim', 'best', 'thing', 'smoothens', 'skin', 'make', 'soft'],
 ['using', 'combination', 'greasy', 'absorbs', 'quickly', 'well', "doesn't"],
 ['whip',
  'act',
  'great',
  'base',
  'primer',
  'moisturize',
  'felt',
  'needed',
  'moisturizer',
  'worth',
  'price',
  'buying'],
 ['good', 'please', 'refresh', 'try']]

In [15]:
df_raw["review_text"].fillna("").tolist()[:5]

['Works as it claims. Could see the difference from the first day. Use it with Olay cleanser for best results',
 'It does what it claims . Best thing is it smoothens ur skin n makes it soft . I liked it',
 'I have been using this product for months now.. it is perfect for combination n oily skin as it is non greasy absorbs quickly and moisturises well but it doesnt work for winters',
 'i have an oily skin, while this whip acts as a great base or primer as it smoothens the skin but it does not moisturise skin the skin, i felt that i still needed a moisturiser. Not worth the price, not buying it again.',
 "It's not that good. Please refresh try for other products"]

In [16]:
save_vocab(processed_docs, VOCAB_PATH)


[Saved] 9025 words → ./data/vocab.txt


In [17]:
df_final = df_raw.copy() # deep copy
df_final["processed_review_text"] = [" ".join(tokens) for tokens in processed_docs]
df_final.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
print(f"\nSaved to {OUTPUT_PATH}")


Saved to ./data/processed.csv
